In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])

In [ ]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader

DATASET_PATH = "../data/food-101/images"

# CARGAR DATASET CORRECTAMENTE
dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

# CLASES REALES (ORDEN CORRECTO AUTOMÁTICO)
CLASSES = dataset.classes

# DATALOADER OPTIMIZADO
loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total imágenes:", len(dataset))
print("Clases reales:", CLASSES)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Cargar modelo preentrenado
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Reemplazar la última capa (MUY IMPORTANTE)
model.classifier[1] = nn.Linear(1280, len(CLASSES))

# Inicializar bien la nueva capa
nn.init.xavier_uniform_(model.classifier[1].weight)

# FASE 1: congelar todo EXCEPTO el clasificador
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()

#FASE 1: entrenar solo la última capa (rápido)
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

EPOCHS = 5

print("Fase 1: Entrenando clasificador")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[F1] Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")


# FASE 2: fine-tuning (CLAVE REAL)
print("Fase 2: Fine-tuning completo")

# descongelar TODO el modelo
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[F2] Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

In [ ]:
torch.save(model.state_dict(), "../models/modelo_efficientnet.pth")
print("Modelo guardado correctamente")

In [ ]:
import json

ruta = r"C:\Users\brayn\Desktop\DOCS DEV\Clasificador de Alimentos\models\classes.json"

with open(ruta, "w") as f:
    json.dump(CLASSES, f)